# Amazon ML Challenge 2026: Entity Resolution Suite
### Google Colab GPU/CPU Runner with Google Drive Persistence & Auto-Resume

**Running in Antigravity IDE via Google Colab Extension**:
1. Open Command Palette (`Ctrl+Shift+P` / `Cmd+Shift+P`).
2. Select `Google Colab: Connect to a Colab runtime` (or click Kernel selector top-right -> Google Colab).
3. Choose runtime accelerator: **L4 GPU** (or T4 GPU) with High-RAM.
4. Run all cells sequentially!

**Key Pipeline Architecture**:
- **Dedicated Run Isolation**: Each run writes to `output/runs/<run_id>/` and `drive/runs/<run_id>/`—prior outputs are never overwritten!
- **Speed-Optimized Inference**: TF-IDF cosine candidate pre-pruning + batched gatekeeper inference cuts runtime by >65%.
- **Drive Persistence**: Real-time mirroring to Google Drive checkpoints (`/content/drive/MyDrive/amazon-ml-challenge-2026`).
- **Official Macro F0.5 Metric**: Direct threshold optimization weighted towards precision (beta=0.5).
- **Memory-Safe Audit**: Built-in format verification with `utils/validate_submission.py`.

In [1]:
# 1. Verify GPU availability and system resources
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM Capacity:   {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
!nvidia-smi
!lscpu | grep -E "Model name|CPU\(s\):" || true

PyTorch Version: 2.11.0+cu128
CUDA Available:  True
GPU Device:      NVIDIA L4
VRAM Capacity:   22.03 GB
Sun Sep 27 05:37:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   34C    P8             13W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                  

In [3]:
# 2. Mount Google Drive to persist all datasets, weights, and benchmark results
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


In [4]:
# 3. Initialize Google Drive directory structure for amazon-ml-challenge-2026
import os
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/amazon-ml-challenge-2026")
DRIVE_DATA = DRIVE_ROOT / "data"
DRIVE_WEIGHTS = DRIVE_ROOT / "weights"
DRIVE_CHECKPOINTS = DRIVE_ROOT / "checkpoints"
DRIVE_OUTPUT = DRIVE_ROOT / "output"

for folder in [DRIVE_ROOT, DRIVE_DATA, DRIVE_WEIGHTS, DRIVE_CHECKPOINTS, DRIVE_OUTPUT]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"[✓] Google Drive structure initialized at: {DRIVE_ROOT}")
print(f"    - Datasets Dir   : {DRIVE_DATA}")
print(f"    - Weights Dir    : {DRIVE_WEIGHTS}")
print(f"    - Checkpoints Dir: {DRIVE_CHECKPOINTS}")
print(f"    - Output Dir     : {DRIVE_OUTPUT}")

[✓] Google Drive structure initialized at: /content/drive/MyDrive/amazon-ml-challenge-2026
    - Datasets Dir   : /content/drive/MyDrive/amazon-ml-challenge-2026/data
    - Weights Dir    : /content/drive/MyDrive/amazon-ml-challenge-2026/weights
    - Checkpoints Dir: /content/drive/MyDrive/amazon-ml-challenge-2026/checkpoints
    - Output Dir     : /content/drive/MyDrive/amazon-ml-challenge-2026/output


In [5]:
# 4. Clone or pull the repository
import os
REPO_URL = "https://github.com/Suyash8/amazon-ml-challenge.git"
REPO_DIR = "/content/amazon-ml-challenge"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
%cd $REPO_DIR
!git pull || true

Cloning into '/content/amazon-ml-challenge'...
remote: Enumerating objects: 144, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 144 (delta 70), reused 119 (delta 45), pack-reused 0 (from 0)
Receiving objects: 100% (144/144), 168.80 KiB | 18.76 MiB/s, done.
Resolving deltas: 100% (70/70), done.
/content/amazon-ml-challenge
Already up to date.


In [6]:
# 5. Install required packages (rapid string matching, lightgbm, etc.)
!python install.py

 Amazon ML Challenge 2026 - Dependency Installer
[*] 'uv' detected! Using uv pip for high-speed package installation.
Using Python 3.13.15 environment at: /usr
Resolved 18 packages in 973ms                                        
Prepared 2 packages in 274ms                                             
Installed 2 packages in 1ms                                 
 + metaphone==0.6
 + rapidfuzz==3.14.6

[✓] All dependencies successfully installed!


In [7]:
# 6. Dataset Linking, Shards Setup & Verification
# Google Drive datasets (/content/drive/MyDrive/amazon-ml-challenge-2026/data/)
# are symlinked to local data/ for zero-copy high-throughput I/O.
import os
from pathlib import Path

!mkdir -p data
drive_data = Path("/content/drive/MyDrive/amazon-ml-challenge-2026/data")
if drive_data.exists():
    for item in drive_data.iterdir():
        target = Path(f"data/{item.name}")
        if not target.exists():
            os.symlink(str(item), str(target))
            print(f"[✓] Linked {item.name} from Google Drive -> data/{item.name}")

# Check available dataset paths
print("\n--- AVAILABLE DATASETS CHECK ---")
for p in ["data/shards/train", "data/shards/test", "dataset_split/train", "dataset_split/test"]:
    if Path(p).exists():
        print(f"  [READY] Found dataset: {p}")
    else:
        print(f"  [INFO] Path not local: {p} (will look in Google Drive if uploaded)")

[✓] Linked shards from Google Drive -> data/shards

--- AVAILABLE DATASETS CHECK ---
  [READY] Found dataset: data/shards/train
  [READY] Found dataset: data/shards/test
  [READY] Found dataset: dataset_split/train
  [READY] Found dataset: dataset_split/test


In [8]:
# 7. Fast Smoke Test (~10 seconds)
# Verifies normalization, candidate blocking, classifier, decision layer & audit end-to-end
!python scripts/run_pipeline.py \
    --train-dir dataset_split/train \
    --test-dir dataset_split/test \
    --drive-sync-dir /content/drive/MyDrive/amazon-ml-challenge-2026 \
    --smoke-test

   AMAZON ML CHALLENGE 2026 - COLAB & GPU HIGH-PERFORMANCE RUNNER
Run ID:            smoke_test_20260927_053948
Device Mode:       CUDA (NVIDIA L4)
CPU Workers:       12 threads (Configured n_jobs: -1)
System RAM:        54.47 GB free / 56.86 GB total
Train Dataset:     /content/amazon-ml-challenge/dataset_split/train
Test Dataset:      /content/amazon-ml-challenge/dataset_split/test
Run Output Dir:    /content/amazon-ml-challenge/output/runs/smoke_test_20260927_053948
Run Checkpoints:   /content/amazon-ml-challenge/checkpoints/runs/smoke_test_20260927_053948 (Resume: True)
Google Drive Sync: /content/drive/MyDrive/amazon-ml-challenge-2026 (runs/smoke_test_20260927_053948)
Smoke Test:        True
----------------------------------------------------------------------------

[Stage 1/3] Training Entity Resolution Pipeline...
  * Reading training ground truth from /content/amazon-ml-challenge/dataset_split/train/train_ground_truth.tsv...
  * Streaming training sources for 29 S1 entities (

In [ ]:
# 9. Full Accelerated Pipeline Execution (GPU-Accelerated with Real-Time Drive Sync)
# Automatically updates to the latest code, utilizes GPU VRAM on NVIDIA L4, and executes at ~2.5s/batch!

import os
import time

# 1. Ensure latest code is loaded from GitHub
%cd /content/amazon-ml-challenge
!git fetch origin main
!git reset --hard origin/main

train_path = "data/shards/train" if os.path.exists("data/shards/train") else "dataset_split/train"
test_path = "data/shards/test" if os.path.exists("data/shards/test") else "dataset_split/test"

run_id = f"prod_l4_turbo_{time.strftime('%Y%m%d_%H%M%S')}"
print(f"Starting Run ID: {run_id}")
print(f"Train Dataset:   {train_path}")
print(f"Test Dataset:    {test_path}")

# 2. Reuse pre-computed candidate_pairs.tsv if available (skips 25-min blocking)
cands_arg = ""
for c_path in [
    "/content/drive/MyDrive/amazon-ml-challenge-2026/output/candidate_pairs.tsv",
    "/content/drive/MyDrive/amazon-ml-challenge-2026/latest/candidate_pairs.tsv",
    "output/runs/run_full_production_v1/candidate_pairs.tsv",
]:
    if os.path.isfile(c_path):
        print(f"[*] Found pre-computed candidates: {c_path}")
        cands_arg = f"--candidate-pairs-path {c_path}"
        break

# 3. Reuse trained model if available (skips 6.5-min Stage 1 training)
model_arg = ""
for m_path in [
    "/content/drive/MyDrive/amazon-ml-challenge-2026/runs/prod_l4_fast_20260927_062442/checkpoints/lgbm_entity_resolver.pkl",
    "/content/amazon-ml-challenge/checkpoints/runs/prod_l4_fast_20260927_062442/lgbm_entity_resolver.pkl",
    "/content/drive/MyDrive/amazon-ml-challenge-2026/checkpoints/lgbm_entity_resolver.pkl",
]:
    if os.path.isfile(m_path):
        print(f"[*] Found pre-trained model: {m_path} (skipping Stage 1 re-training)")
        model_arg = f"--model-path {m_path}"
        break

# 4. Launch turbo runner with active GPU VRAM buffer and parallel workers
!python scripts/run_pipeline.py \
    --train-dir $train_path \
    --test-dir $test_path \
    --run-id $run_id \
    --drive-sync-dir /content/drive/MyDrive/amazon-ml-challenge-2026 \
    --device auto \
    --batch-size 10000 \
    --n-jobs -1 \
    $cands_arg \
    $model_arg


Starting Run ID: prod_l4_fast_20260927_053959
Train Dataset:   data/shards/train
Test Dataset:    data/shards/test
[*] Found pre-computed candidate_pairs.tsv in Google Drive: /content/drive/MyDrive/amazon-ml-challenge-2026/output/candidate_pairs.tsv
[*] Linking pre-computed candidates from /content/drive/MyDrive/amazon-ml-challenge-2026/output/candidate_pairs.tsv to /content/amazon-ml-challenge/output/runs/prod_l4_fast_20260927_053959/candidate_pairs.tsv...
   AMAZON ML CHALLENGE 2026 - COLAB & GPU HIGH-PERFORMANCE RUNNER
Run ID:            prod_l4_fast_20260927_053959
Device Mode:       CUDA (NVIDIA L4)
CPU Workers:       12 threads (Configured n_jobs: -1)
System RAM:        54.39 GB free / 56.86 GB total
Train Dataset:     /content/drive/MyDrive/amazon-ml-challenge-2026/data/shards/train
Test Dataset:      /content/drive/MyDrive/amazon-ml-challenge-2026/data/shards/test
Run Output Dir:    /content/amazon-ml-challenge/output/runs/prod_l4_fast_20260927_053959
Run Checkpoints:   /conten

In [ ]:
# 10. Run Official Submission Validator on Entire Output & Save Audit Report
# Validates format, ID compliance, entity completeness, and required columns for competition submission (<50MB RAM, ~15s)
import os
import sys
import shutil
import subprocess
from pathlib import Path

# Optional: manually set a specific path, or leave None for automatic newest-run detection
MATCHING_FILE_OVERRIDE = None

# Locate test dataset directory
test_candidates = [
    Path("data/shards/test"),
    Path("dataset_split/test"),
    Path("/content/drive/MyDrive/amazon-ml-challenge-2026/data/shards/test"),
    Path("/content/drive/MyDrive/amazon-ml-challenge-2026/data/test"),
    Path("/content/drive/MyDrive/amazon-ml-challenge-2026/data"),
    Path("data/test"),
    Path("dataset/test"),
]
test_path = None
for p in test_candidates:
    if p.exists() and (
        (p / "test_source1").exists()
        or (p / "test_source1.tsv").exists()
        or (p / "test_source1.tsv.gz").exists()
    ):
        test_path = p
        break

if test_path is None:
    test_path = Path("data/shards/test") if os.path.exists("data/shards/test") else Path("dataset_split/test")

# Collect all potential matching_results.tsv candidates
matching_candidates = [
    Path("output/latest/matching_results.tsv"),
    Path("output/matching_results.tsv"),
    Path("/content/drive/MyDrive/amazon-ml-challenge-2026/latest/matching_results.tsv"),
    Path("/content/drive/MyDrive/amazon-ml-challenge-2026/output/latest/matching_results.tsv"),
    Path("/content/drive/MyDrive/amazon-ml-challenge-2026/output/matching_results.tsv"),
]

# Search in output/runs/*/
runs_dir = Path("output/runs")
if runs_dir.is_dir():
    matching_candidates.extend(runs_dir.glob("*/matching_results.tsv"))

# Search in output/gdrive*/
matching_candidates.extend(Path("output").glob("gdrive*/matching_results.tsv"))

# Search in Google Drive runs/
drive_runs = Path("/content/drive/MyDrive/amazon-ml-challenge-2026/runs")
if drive_runs.is_dir():
    matching_candidates.extend(drive_runs.glob("*/matching_results.tsv"))

if MATCHING_FILE_OVERRIDE and Path(MATCHING_FILE_OVERRIDE).is_file():
    matching_file = Path(MATCHING_FILE_OVERRIDE)
else:
    # Filter to existing non-empty files and select the newest by modification time
    valid_matches = [p for p in matching_candidates if p.is_file() and p.stat().st_size > 0]
    if not valid_matches:
        raise FileNotFoundError(
            "Could not locate any non-empty matching_results.tsv to validate!\n"
            "Please run Cell 9 to produce inference output."
        )
    matching_file = max(valid_matches, key=lambda p: p.stat().st_mtime)

# Locate candidate_pairs.tsv corresponding to this run
candidate_file = None
possible_cands = [
    matching_file.parent / "candidate_pairs.tsv",
    matching_file.parent / "candidate_pairs.txt",
    Path("/content/drive/MyDrive/amazon-ml-challenge-2026/output/candidate_pairs.tsv"),
    Path("/content/drive/MyDrive/amazon-ml-challenge-2026/latest/candidate_pairs.tsv"),
    Path("/content/drive/MyDrive/amazon-ml-challenge-2026/candidate_pairs.tsv"),
    Path("/content/drive/MyDrive/amazon-ml-challenge-2026/output/candidate_pairs.txt"),
    Path("output/latest/candidate_pairs.tsv"),
    Path("output/candidate_pairs.tsv"),
]
for cp in possible_cands:
    if cp.is_file() and cp.stat().st_size > 0:
        # If matching file is full-scale (>10MB), don't pair it with a tiny smoke-test candidate sample (<10MB)
        if matching_file.stat().st_size > 10_000_000 and cp.stat().st_size < 10_000_000:
            continue
        candidate_file = cp
        break

print("=" * 76)
print("   AMAZON ML CHALLENGE 2026 — SUBMISSION AUDIT & VALIDATION")
print("=" * 76)
print(f"[*] Matching file:     {matching_file.resolve()} ({matching_file.stat().st_size / 1024 / 1024:.2f} MB)")
print(f"[*] Candidate file:    {candidate_file.resolve() if candidate_file else '[Not found - skipped]'}")
print(f"[*] Test data dir:     {test_path.resolve()}")

# 1. Full Stream Audit of the Entire Matching File (all rows)
line_count = 0
non_empty = 0
empty_singletons = 0
total_matches = 0
seen_ids = set()
duplicate_ids = 0

with open(matching_file, "r", encoding="utf-8") as f:
    first_line = f.readline()
    header = [col.strip().lower() for col in first_line.rstrip("\r\n").split("\t")]
    expected_header = ["source1_entity_id", "matched_entity_ids"]
    if header != expected_header:
        print(f"[!] WARNING: Header is {header}, expected {expected_header}")
    
    for line in f:
        line_str = line.strip()
        if not line_str:
            continue
        line_count += 1
        parts = line_str.split("\t")
        s1 = parts[0].strip()
        if s1 in seen_ids:
            duplicate_ids += 1
        else:
            seen_ids.add(s1)
            
        if len(parts) > 1 and parts[1].strip():
            matched_ids = [m.strip() for m in parts[1].split(",") if m.strip()]
            non_empty += 1
            total_matches += len(matched_ids)
        else:
            empty_singletons += 1

print("\n[1] File Integrity Summary:")
print(f"    - Total Entities (Rows): {line_count:,} (Expected: 1,732,544)")
print(f"    - Non-empty Predictions: {non_empty:,} ({non_empty/max(1, line_count)*100:.2f}%)")
print(f"    - Singleton Predictions: {empty_singletons:,} ({empty_singletons/max(1, line_count)*100:.2f}%)")
if non_empty > 0:
    print(f"    - Total Matched Pairs:   {total_matches:,} (Avg {total_matches/non_empty:.2f} matches/entity)")
print(f"    - Duplicate S1 IDs:      {duplicate_ids}")

# 2. Run Official Competition Validator
# Explicitly pass candidate path or NONE so validator doesn't accidentally load unrelated candidate files
cand_arg = ["--candidate", str(candidate_file) if candidate_file else "NONE"]
cmd = [sys.executable, "utils/validate_submission.py", "--matching", str(matching_file), "--test-dir", str(test_path)] + cand_arg
res = subprocess.run(cmd, capture_output=True, text=True)

print("\n[2] Official Validator Execution:")
print(res.stdout)
if res.stderr:
    print("STDERR:", res.stderr)

# 3. Save Comprehensive Audit Report
report_text = f"""=== SUBMISSION VALIDATION AUDIT REPORT ===
Matching File:    {matching_file.resolve()}
Candidate File:   {candidate_file.resolve() if candidate_file else 'None'}
Test Directory:   {test_path.resolve()}

--- METRICS & COUNTS ---
Total Test Rows:  {line_count:,}
Non-Empty Rows:   {non_empty:,} ({non_empty/max(1, line_count)*100:.2f}%)
Singleton Rows:   {empty_singletons:,} ({empty_singletons/max(1, line_count)*100:.2f}%)
Total Matches:    {total_matches:,}
Duplicate IDs:    {duplicate_ids}
Exit Code:        {res.returncode}

--- OFFICIAL VALIDATOR OUTPUT ---
{res.stdout}

--- VALIDATOR STDERR ---
{res.stderr if res.stderr else '[None]'}
"""

save_destinations = [
    Path("output/latest/validation_report.txt"),
    matching_file.parent / "validation_report.txt",
    Path("/content/drive/MyDrive/amazon-ml-challenge-2026/output/latest/validation_report.txt"),
    Path("/content/drive/MyDrive/amazon-ml-challenge-2026/latest/validation_report.txt"),
    Path("/content/drive/MyDrive/amazon-ml-challenge-2026/output/validation_report.txt"),
]

saved_paths = []
for dest in save_destinations:
    try:
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_text(report_text, encoding="utf-8")
        saved_paths.append(str(dest.resolve()))
    except Exception:
        pass

print("\n[3] Validation Audit Report Saved to:")
for sp in saved_paths:
    print(f"    - {sp}")

if res.returncode == 0:
    print("\n" + "=" * 76)
    print("   ✅ SUBMISSION VALIDATION PASSED: 100% COMPLIANT WITH COMPETITION RULES!")
    print("   File is safe to submit to portal.")
    print("=" * 76)
else:
    print("\n" + "=" * 76)
    print(f"   ❌ SUBMISSION VALIDATION FAILED (Exit Code: {res.returncode})")
    print("   Please review the errors listed above before submitting.")
    print("=" * 76)


In [ ]:
# 11. Display Scorecard & Submission Preview Inline
import json
from pathlib import Path
from IPython.display import display, Markdown

scorecard_candidates = [
    Path("output/latest/scorecard.json"),
    Path("output/scorecard.json"),
    Path("/content/drive/MyDrive/amazon-ml-challenge-2026/output/latest/scorecard.json"),
    Path("/content/drive/MyDrive/amazon-ml-challenge-2026/latest/scorecard.json"),
    Path("/content/drive/MyDrive/amazon-ml-challenge-2026/output/scorecard.json"),
]
# Search in output/runs/*/scorecard.json
runs_dir = Path("output/runs")
if runs_dir.is_dir():
    scorecard_candidates.extend(runs_dir.glob("*/scorecard.json"))

valid_sc = [p for p in scorecard_candidates if p.is_file() and p.stat().st_size > 0]
if valid_sc:
    newest_sc = max(valid_sc, key=lambda p: p.stat().st_mtime)
    print(f"[*] Displaying scorecard from: {newest_sc}")
    sc = json.loads(newest_sc.read_text(encoding="utf-8"))
    md_table = f"""
### 📊 Evaluation Scorecard (3 Pipeline Tiers)
| Pipeline Tier | Metric | Value |
| :--- | :--- | :--- |
| **1. Candidate Blocking** | Pair Recall | **{sc.get('blocking_pair_recall', 0)*100:.2f}%** |
| | Space Reduction Rate | **{sc.get('candidate_reduction_rate', 0)*100:.2f}%** |
| | Candidates / Entity | **{sc.get('candidates_per_entity', 0):.1f}** |
| **2. GBDT Classifier** | Pair Precision | **{sc.get('classifier_pair_precision', 0)*100:.2f}%** |
| | Pair Recall | **{sc.get('classifier_pair_recall', 0)*100:.2f}%** |
| | Pairwise F1 | **{sc.get('classifier_pair_f1', 0):.4f}** |
| **3. Decision Layer** | **Official Macro F0.5** | **{sc.get('official_macro_f05', 0):.4f}** |
| | Singleton Sub-Score | **{sc.get('singleton_macro_f05', 0):.4f}** |
| | Matched Sub-Score | **{sc.get('matched_macro_f05', 0):.4f}** |
"""
    display(Markdown(md_table))

print("\n--- Latest Run Output Files ---")
!ls -lh output/latest/ 2>/dev/null || ls -lh output/
print("\n--- First 5 rows of matching_results.tsv ---")
!head -n 6 output/latest/matching_results.tsv 2>/dev/null || head -n 6 output/matching_results.tsv
